# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_dict = json.loads(dataset.metadata.to_json())
print(f"{metadata_dict['name']} (v{metadata_dict['version']})\n")
print(metadata_dict['description'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available Record Sets and their @ids
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

if len(record_sets) > 0:
    # Show fields (and columns) of the first record set
    rs_id = record_sets[0]['@id']
    fields = dataset.fields(record_set=rs_id)
    print(f"\nFields for RecordSet @id '{rs_id}':")
    for field in fields:
        print(f"  Field @id: {field['@id']} | name: {field.get('name', '(no name)')} | type: {field.get('dataType', '')}")
        # If the field has column information, print column id
        if 'column' in field:
            col = field['column']
            if isinstance(col, dict):
                col_id = col.get('@id', str(col))
            else:
                col_id = str(col)
            print(f"    Corresponding Column @id: {col_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set

# Collect record_set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = dict()

for rs_id in record_set_ids:
    # Each record set yields dict records
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns for the first record set as an example
if len(record_set_ids) > 0:
    print(f"DataFrame columns for RecordSet @id '{record_set_ids[0]}':")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, let's pick the first record set (if any exist); update as appropriate for your analysis
if len(record_set_ids) > 0:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    print(f"\nInspecting first 5 rows of RecordSet @id {rs_id}...")
    print(df.head())

    # List numeric columns (float/int)
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if len(numeric_cols):
        numeric_field_id = numeric_cols[0]
        print(f"\nNumeric field candidates: {numeric_cols}\nUsing '{numeric_field_id}' for filtering and normalization.")

        threshold = df[numeric_field_id].mean()  # Example: use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (total: {filtered_df.shape[0]} records):")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Find candidate for group field (categorical)
        object_cols = df.select_dtypes(include=['object']).columns.tolist()
        if object_cols:
            group_field_id = object_cols[0]
            print(f"\nGrouping by field '{group_field_id}'")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped means:")
            print(grouped_df.head())
    else:
        print("No numeric fields detected in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution if possible
if len(record_set_ids) > 0:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id].copy()
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        col = numeric_cols[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[col].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of '{col}' in RecordSet {rs_id}")
        plt.xlabel(col)
        plt.show()

    # Scatter plot with first two numeric fields if possible
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(6,6))
        sns.scatterplot(data=df, x=numeric_cols[0], y=numeric_cols[1])
        plt.title(f"Scatter plot of {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and explored the FAIR\(^2\) dataset describing mass spectrometry results of extracellular vesicles from human mast cells. Using the `mlcroissant` library, we:
- Loaded dataset metadata and inspected its description,
- Explored the available record sets, fields, and their Croissant `@id`s,
- Loaded tabular records into pandas DataFrames,
- Filtered, normalized, and grouped data based on field types,
- Visualized numerical distributions and bivariate relationships.

Please refer to the dataset's [source on sen.science](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and its Croissant schema for further in-depth analyses and machine learning applications.